In [1]:
import pickle
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from imblearn.over_sampling import SMOTE
import mlflow
import warnings
warnings.filterwarnings('ignore')

with open('../src/features/X_y.pkl', 'rb') as f:
    X, y = pickle.load(f)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Before SMOTE: Fake={y_train.sum()}, Real={len(y_train)-y_train.sum()}")

c:\Users\MH Mazin\fake-job-detector\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Before SMOTE: Fake=693, Real=13611


In [2]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"After SMOTE: Fake={y_train_sm.sum()}, Real={len(y_train_sm)-y_train_sm.sum()}")

After SMOTE: Fake=13611, Real=13611


In [3]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fake_job_detection_v2")

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    }

2026/07/08 23:40:34 INFO mlflow.tracking.fluent: Experiment with name 'fake_job_detection_v2' does not exist. Creating a new experiment.


In [4]:
with mlflow.start_run(run_name="lr_smote"):
    params = {"C": 1.0, "max_iter": 1000, "class_weight": "balanced"}
    mlflow.log_params(params)
    
    lr = LogisticRegression(**params)
    lr.fit(X_train_sm, y_train_sm)
    
    m = evaluate(lr, X_test, y_test)
    mlflow.log_metrics(m)
    print(f"LR+SMOTE - F1: {m['f1']:.3f}, Recall: {m['recall']:.3f}")

LR+SMOTE - F1: 0.373, Recall: 0.757


In [5]:
with mlflow.start_run(run_name="rf_smote"):
    params = {"n_estimators": 200, "max_depth": 15, "class_weight": "balanced", "random_state": 42}
    mlflow.log_params(params)
    
    rf = RandomForestClassifier(**params)
    rf.fit(X_train_sm, y_train_sm)
    
    m = evaluate(rf, X_test, y_test)
    mlflow.log_metrics(m)
    print(f"RF+SMOTE - F1: {m['f1']:.3f}, Recall: {m['recall']:.3f}")

RF+SMOTE - F1: 0.417, Recall: 0.607


In [6]:
from mlflow.tracking import MlflowClient
client = MlflowClient()
exp = client.get_experiment_by_name("fake_job_detection_v2")
runs = client.search_runs(experiment_ids=[exp.experiment_id])

for run in runs:
    print(f"{run.info.run_name}: F1={run.data.metrics.get('f1', 0):.3f}")

best = max(runs, key=lambda r: r.data.metrics.get("f1", 0))
print(f"\nBest: {best.info.run_name}")

rf_smote: F1=0.417
lr_smote: F1=0.373

Best: rf_smote


In [8]:
import os
os.makedirs('../src/models', exist_ok=True)

# Save best model
with open('../src/models/best_model.pkl', 'wb') as f:
    pickle.dump(lr if 'lr' in best.info.run_name else rf, f)

# Save vectorizer too
import shutil
shutil.copy('../src/features/tfidf_vectorizer.pkl', '../src/models/vectorizer.pkl')

print("Model and vectorizer saved to src/models/")

Model and vectorizer saved to src/models/
